In [1]:
import torch
import myneural as mn
import torch.nn.functional as F
import time
import pyarrow.parquet as pq
import sentencepiece as spm
import math
import os
torch.manual_seed(42)

In [2]:
TOKENIZER_DIR = "models/my_tokenizer.model"

config = mn.Config()
config.n_embd = 256
config.n_head = 8
config.n_layer = 8
config.mini_batch_size = 16 # B
config.context_length = 2048 # T
config.total_batch_size = 4 * config.mini_batch_size * config.context_length # MUST BE DIVISBLE by (B*T)
config.device = 'cuda'
config.vocab_size = 32768

max_iter = 20000
B = config.mini_batch_size
T = config.context_length
grad_accum_steps = config.total_batch_size // (B*T)
assert config.total_batch_size % (B * T) == 0, "Total batch size must be divisble by (B*T)"

In [3]:
import remotedataset as rd
import sentencepiece as spm

if not os.path.exists(TOKENIZER_DIR):
    hf_dataset = rd.get_interleaved_dataset()
    rd.train_tokenizer(rd.create_tokenizer_corpus(hf_dataset)) 
    
sp = spm.SentencePieceProcessor(model_file = TOKENIZER_DIR) # pyright: ignore[reportCallIssue]

In [4]:
hf_dataset = rd.get_interleaved_dataset()
val_hf = hf_dataset.take(1000)
train_hf = hf_dataset.skip(1000)

train_ds = rd.PackedDataset(train_hf, sp, config.mini_batch_size, config.context_length)
val_ds = rd.PackedDataset(val_hf, sp, config.mini_batch_size, config.context_length)
train_dataloader = torch.utils.data.DataLoader(train_ds, batch_size=None, num_workers=0, pin_memory=True)#, prefetch_factor=4, persistent_workers=True)
val_dataloader = torch.utils.data.DataLoader(val_ds, batch_size=None, num_workers=0, pin_memory=True)#, prefetch_factor=4, persistent_workers=True)

Resolving data files:   0%|          | 0/104 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/104 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/104 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/234 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/184 [00:00<?, ?it/s]

In [5]:
print(sp.Encode("Lily found a needle.")) # pyright: ignore[reportAttributeAccessIssue]
print(sp.Decode(sp.Encode("Lily found a needle.")))

[7006, 1189, 267, 12363, 32692]
Lily found a needle.


In [6]:
def get_lr(step):
    max_lr = 1e-3
    min_lr = 1e-4
    warmup_steps=200
    max_steps=20000
    if step < warmup_steps:
        return max_lr * (step+1)/warmup_steps 
    if step > max_steps:
        return min_lr
    decay_ratio = (step-warmup_steps)/(max_steps-warmup_steps)
    coefficient = 0.5 * (1.0 + math.cos(decay_ratio * math.pi)) # starts at 1 reduces to 0
    return min_lr + coefficient * (max_lr - min_lr)
    

In [7]:
torch.set_float32_matmul_precision('high')
model = mn.Transformer(config, dtype=torch.float32, device=config.device)
params = model.parameters()

In [8]:
optimizer_adamw = model.configure_optimizer(weight_decay=0.1, learning_rate=get_lr(0), device=config.device)

In [9]:
# # Load the list of tensors
# loaded_params = torch.load("models/params_new.pth")

# # Manually re-assign them to your model
# for p, loaded_p in zip(model.parameters(), loaded_params):
#     p.data.copy_(loaded_p.data)

In [10]:
model_optim = torch.compile(model)
step = 0

In [14]:
def get_val_loss():
    max_batches = 10
    val_loss = []
    with torch.no_grad():
        for i, (xs, ys) in enumerate(val_dataloader):      
            if i >= max_batches:
                break
            xs = xs.to(device=config.device, non_blocking=True)
            ys= ys.to(device=config.device, non_blocking=True)
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                _, loss = model_optim(xs, targets=ys)
            val_loss.append(loss.item())
    return sum(val_loss)/len(val_loss)

In [16]:
data_iter = iter(train_dataloader)
lossi = []
timei = []
mean_stat_batches = 1
print_batches = 1
val_batches = 100
for _ in range(step, max_iter):
    t_start = time.time()
    optimizer_adamw.zero_grad(set_to_none=True)
    # optimizer_muon.zero_grad(set_to_none=True)
    # forward pass
    loss_accum = torch.tensor(0.0, device=config.device)
    for micro_step in range(grad_accum_steps):
        xs, ys= next(data_iter) # loop_around=True so we don't need to worry about StopIteration exception
        xs = xs.to(device=config.device, non_blocking=True)
        ys= ys.to(device=config.device, non_blocking=True)
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            logits, loss = model_optim(xs, targets=ys)
        loss = loss / grad_accum_steps # normalizer to avoid loss gradients being exagerrated by grad_accum_steps times.
        loss_accum += loss.detach()
        # backward pass
        loss.backward()
    
    norm = torch.nn.utils.clip_grad_norm_(params, 1.0)
    
    # update
    lr = get_lr(step)
    for param_group in optimizer_adamw.param_groups:
        param_group['lr'] = lr
    optimizer_adamw.step()
    
    # for param_group in optimizer_muon.param_groups:
    #     param_group['lr'] = lr
    # optimizer_muon.step()
    
    torch.cuda.synchronize()
    t_end = time.time()
    
    # track stats    
    lossi.append(loss_accum.item())
    timei.append(t_end-t_start)
    if step % print_batches == 0:
        lossi = lossi[-mean_stat_batches:]
        timei = timei[-mean_stat_batches:]
        train_loss = sum(lossi)/len(lossi)
        if step % val_batches == 0:
            val_loss = get_val_loss()
        else: 
            val_loss = 0.0
        dt = sum(timei)/len(timei)
        tokens_per_sec = (xs.shape[0] * xs.shape[1] * grad_accum_steps)/dt # (B*T)/dt
        print(f"Step {step}: Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | dt: {dt*1000:.2f}ms | {int(tokens_per_sec):,d} tok/s")

    step += 1

Step 62: Train Loss: 8.0218 | Val Loss: 0.0000 | dt: 54262.38ms | 2,415 tok/s
Step 63: Train Loss: 7.9238 | Val Loss: 0.0000 | dt: 6767.85ms | 19,366 tok/s
Step 64: Train Loss: 7.7652 | Val Loss: 0.0000 | dt: 7191.97ms | 18,224 tok/s
Step 65: Train Loss: 7.8826 | Val Loss: 0.0000 | dt: 7357.31ms | 17,815 tok/s
Step 66: Train Loss: 7.9036 | Val Loss: 0.0000 | dt: 7545.81ms | 17,370 tok/s
Step 67: Train Loss: 7.4396 | Val Loss: 0.0000 | dt: 7217.29ms | 18,160 tok/s
Step 68: Train Loss: 7.8078 | Val Loss: 0.0000 | dt: 7727.21ms | 16,962 tok/s
Step 69: Train Loss: 7.8105 | Val Loss: 0.0000 | dt: 9539.30ms | 13,740 tok/s
Step 70: Train Loss: 7.8034 | Val Loss: 0.0000 | dt: 7772.78ms | 16,862 tok/s
Step 71: Train Loss: 7.6889 | Val Loss: 0.0000 | dt: 7067.86ms | 18,544 tok/s
Step 72: Train Loss: 7.7219 | Val Loss: 0.0000 | dt: 6751.22ms | 19,414 tok/s
Step 73: Train Loss: 7.7301 | Val Loss: 0.0000 | dt: 6533.14ms | 20,062 tok/s
Step 74: Train Loss: 7.7524 | Val Loss: 0.0000 | dt: 6195.97ms |

KeyboardInterrupt: 

In [17]:
val_loss = get_val_loss()
val_loss

: 

In [ ]:
SAVE = True
if SAVE:
    params_list = list(model.parameters())
    torch.save(params_list, "models/kaggle_test.pth")

In [ ]:
print(mn.prompt("Tim loves to eat toast. So he asked his mom to cook some", model=model, tokenizer=sp, new_tokens=256, debug=True, device=config.device))

</s>
 Tim loves to eat toast. So he asked his mom to cook some more jam on the stove. His mom said, "You should read cows to be good, Tim." Tim quietly put on his toast, and sets the chair behind his back. He pebbles on the big bowl and pretends it is fuel for the itch. With his mom's help, they scrubber the bowl and put it back in the bowl. It was very hot now! Tim closed his eyes and tried to keep the inside, but it was stuck. He asked his mom, "Mom, why is such the m sandwiches falling from the consequences?" His mom smiled and said, "Sometimes the other food that it is stuck outside on the stove, but with shopping things." Tim nodded. He knew that he would be humble just like struggling to find the food rain him when he read.</s>
 Once upon a time there were two friends, Jack and Fred. They were playing in the garden. They were exploring when suddenly Bob heard that they were confused. He said to Fred, "Coming Prunes!" Fred replied, "Watch me, I'm going to please!" Fred smiled and 